# 05 · One `evosax` notebook, any `evosax` algorithm

**What this notebook is.** A small, friendly starting point for using
[`evosax`](https://github.com/RobertTLange/evosax) to search origami fold
angles. You change **one word** — the `STRATEGY` variable — and the same loop
runs a different algorithm. Nothing here re-implements an optimizer: `evosax`
(an evolution-strategy library written in JAX) does the searching, and
`origami-engine-mjx` supplies the *design space*, the *simulation* and the
*fitness*.

**What you will do**

1. describe the fold angles of a tiny paper pattern as a list of numbers
   (a "genome");
2. score a genome with two numbers at once — how far the paper **moves** and how
   well it **folds to its target angles** — and mix them into a single score;
3. run any `evosax` strategy through the same `ask → evaluate → tell` loop;
4. switch `STRATEGY` to compare algorithms;
5. save the best design as a normal crease-pattern file.

> ### New to these words?
>
> * **Genome** — one candidate design: one fold angle per hinge.
> * **Fitness** — one number we want to get big (or small).
> * **Generation** — one round of: ask for a batch of candidates, score them,
>   tell the algorithm how they did.
>
> Every cell is short and prints what it produced, so run them top to bottom.
The list of strategies, the paper, the score and the search loop are the same
cells for every algorithm; only `STRATEGY` changes.

## Install

`evosax` is not a dependency of the library; this notebook installs it
alongside the JAX extra. You only need to do this once per Colab session.

The package comes from the main repository,
[https://github.com/ihsan31415/origami-engine-mjx](https://github.com/ihsan31415/origami-engine-mjx), pinned to a fixed commit
(swap `@957cc57` for `@main` to follow the latest code).

In [ ]:
%pip install -q "origami-engine-mjx[jax] @ git+https://github.com/ihsan31415/origami-engine-mjx.git@957cc57"
%pip install -q "evosax>=0.3"

## 0. Choose the algorithm

All the settings you are likely to change are here.

* `STRATEGY` — which `evosax` algorithm to run. Try `"CMA_ES"` first; it is the
  most reliable default. Any name from `evosax.algorithms.algorithms` works
  (the next cell lists the full set).
* `MOVEMENT_WEIGHT` / `FOLD_WEIGHT` — how to mix the two scores. They are
  **relative**, so `70` / `30` means "70 % movement, 30 % folding". Set
  `FOLD_WEIGHT = 0` to optimise movement only (the old behaviour).
* `POP_SIZE` / `GENERATIONS` — the search budget. Every generation costs
  `POP_SIZE` simulations, so keep both small on a CPU.

In [ ]:
# ---------------------------------------------------------------------
# 1. The one knob that selects the algorithm.
#    Try: "CMA_ES", "Open_ES", "xNES", "PGPE", "ARS", "SimpleGA",
#         "PSO", "DifferentialEvolution", "GuidedES", "Sep_CMA_ES"
# ---------------------------------------------------------------------
STRATEGY = "CMA_ES"

# 2. How to mix the two fitness parts (relative weights).
MOVEMENT_WEIGHT = 70.0  # % of the score from paper movement
FOLD_WEIGHT = 30.0  # % of the score from folding to the target angles

# 3. Search budget. One generation = POP_SIZE simulations.
POP_SIZE = 4
GENERATIONS = 3
SEED = 0

# 4. Simulation length. Small on purpose: a CPU simulation is slow.
EVAL_DURATION = 0.02  # simulated seconds per evaluation
EVAL_FOLD = 0.015  # seconds spent ramping the hinges to their targets

print(f"strategy={STRATEGY}  movement={MOVEMENT_WEIGHT:g}%  fold={FOLD_WEIGHT:g}%")
print(f"pop={POP_SIZE}  generations={GENERATIONS}  -> {POP_SIZE * GENERATIONS} simulations")

## 1. Imports and the list of available algorithms

`evosax.algorithms.algorithms` is a dictionary of every strategy `evosax`
ships. We print it so you can copy any name into `STRATEGY` above.

In [ ]:
import importlib.metadata
import inspect

import jax
import numpy as np
from evosax.algorithms import algorithms as EVO_ALGORITHMS

from origami_engine_mjx import Crease
from origami_engine_mjx.integrations import DesignSpace, OrigamiEvaluator, save_best, save_genome

if STRATEGY not in EVO_ALGORITHMS:
    raise ValueError(f"{STRATEGY!r} is not an evosax strategy. Pick one of: {sorted(EVO_ALGORITHMS)}")

print("jax      :", jax.__version__, "| devices:", jax.devices())
print("evosax   :", importlib.metadata.version("evosax"))
print("strategies:", len(EVO_ALGORITHMS))
print(", ".join(sorted(EVO_ALGORITHMS)))

## 2. Describe the paper

We use a tiny **2×2 Miura** pattern: six crease lines that form a small
zig-zag. Four of those creases act as **hinges** that we can drive; the other
two close a loop and are enforced by constraints.

`DesignSpace.angles_only(pattern)` keeps that shape fixed and lets only the
hinge **angles** change. Each genome is therefore a short list of angles — one
per hinge — and the space knows the smallest and largest sensible value for
each.

In [ ]:
pattern = [
    Crease((0.1, 0.1), (0.1, 0.5), 0.8),
    Crease((0.1, 0.5), (0.5, 0.5), -0.8),
    Crease((0.5, 0.5), (0.5, 0.1), 0.8),
    Crease((0.5, 0.1), (0.1, 0.1), -0.8),
    Crease((0.1, 0.1), (0.5, 0.5), -0.8),
    Crease((0.1, 0.5), (0.5, 0.1), 0.8),
]
space = DesignSpace.angles_only(pattern, name="miura-2x2")

print(f"pattern {space.name!r}")
print(f"genome length (hinges) : {space.dim}")
print(f"lower bounds (radians) : {np.round(space.lower, 3).tolist()}")
print(f"upper bounds (radians) : {np.round(space.upper, 3).tolist()}")

### The `[-1, 1]` view

Algorithms prefer a simple box to work in, so the space also has a
**normalized** view: every genome maps to a vector where each entry lies in
`[-1, 1]`. `evosax` searches there; `space.from_normalized(...)` turns a
candidate back into real fold angles.

```text
genome       [0.8, -0.8, 0.8, -0.8]      real angles (radians)
                  ↕  space.to_normalized / space.from_normalized
normalized   [1.0, -1.0, 1.0, -1.0]      what the algorithm moves around
```

In [ ]:
base = space.base_vector()  # the pattern exactly as written above
z = space.to_normalized(base)
print("base genome     :", np.round(base, 3).tolist())
print("normalized view :", np.round(z, 3).tolist())
assert np.allclose(space.from_normalized(z), base, atol=1e-6)

# A random candidate, already clipped into the box:
print("random candidate:", np.round(space.from_normalized(np.array([0.3, -0.9, 0.5, -0.2])), 3).tolist())

## 3. Score a design: movement **and** folding

`OrigamiEvaluator` runs one real simulation and returns a fitness number.
Two ingredients:

| Ingredient | Question it answers |
|---|---|
| **Movement** | How far did the paper slide across the table? (metres, +x) |
| **Folding** | Did every hinge reach the angle it was designed for? (`0..1`) |

Passing `MOVEMENT_WEIGHT` / `FOLD_WEIGHT` blends them:

```text
score = (movement_weight * movement + fold_weight * fold)
        / (movement_weight + fold_weight)
```

With `FOLD_WEIGHT = 0` you get exactly the old movement score, so nothing
breaks if you do not want folding.

> **Tip:** a fold score of `1.0` means every hinge is within a couple of
> degrees of its target. Watch it climb over generations — that is the paper
> learning to fold itself.

In [ ]:
evaluator = OrigamiEvaluator(
    space,
    backend="jax",
    duration=EVAL_DURATION,
    fold_duration=EVAL_FOLD,
    vibrate=False,  # a steady fold is easier to read than a shaking one
    maximize=True,  # higher fitness = better
    error_policy="penalty",  # a broken candidate gets a very low score, not a crash
    movement_weight=MOVEMENT_WEIGHT,
    fold_weight=FOLD_WEIGHT,
)

print(evaluator)
print(f"movement share: {evaluator.movement_fraction:.0%}   fold share: {evaluator.fold_fraction:.0%}")

# One real simulation, with every number broken out for inspection:
record = evaluator.evaluate(base)
print(f"\nfitness  = {record.fitness:+.4f}")
for key in ("displacement_x", "fold_score", "fold_mean_abs_error_deg"):
    print(f"  {key:24s} {record.descriptors[key]:+.4f}")

## 4. The glue: candidates in, scores out

`evosax` works on normalized vectors and **minimizes**. So we need two small
helpers:

* `to_genome(z)` — clip a normalized vector into `[-1, 1]` and turn it into real
  fold angles;
* `losses_for(Z)` — score a whole batch of them and return `-fitness`
  (negating turns "bigger is better" into "smaller is better").

In [ ]:
def to_genome(z):
    """Normalized candidate -> in-bounds fold angles."""
    return space.from_normalized(np.clip(np.asarray(z, dtype=float), -1.0, 1.0))


def losses_for(Z):
    """Score a batch of normalized candidates; evosax minimizes, so negate."""
    Z = np.clip(np.asarray(Z, dtype=float), -1.0, 1.0)
    records = [evaluator.evaluate(to_genome(row)) for row in Z]
    return np.array([-rec.fitness for rec in records], dtype=float)


Z_demo = np.array([[0.0, 0.0, 0.0, 0.0], [1.0, -1.0, 1.0, -1.0]])
print("demo losses:", np.round(losses_for(Z_demo), 4).tolist())

## 5. Build the chosen algorithm

Different `evosax` strategies need slightly different setup calls — some start
from a single mean, others from a population, and a few from several populations.
Instead of a hard-coded list, `make_strategy` inspects the strategy and picks
the right form, so **every** name in `STRATEGY` works.

In [ ]:
import jax.numpy as jnp


def make_strategy(name, dim, population_size):
    """Create an evosax strategy, giving it any extra arguments it requires."""
    cls = EVO_ALGORITHMS[name]
    solution = jnp.zeros((dim,), dtype=jnp.float32)
    extra = {}
    if name == "RandomSearch":  # needs a proposal rule
        extra["sampling_fn"] = lambda key: jax.random.uniform(key, (dim,), minval=-1.0, maxval=1.0)
    if name in {"SV_CMA_ES", "SV_Open_ES"}:  # Stein-variational variants
        extra["num_populations"] = 2
    return cls(population_size=population_size, solution=solution, **extra)


def start(es, seed):
    """Start the strategy, dispatching on its init() signature."""
    key = jax.random.key(seed)
    key, key_init = jax.random.split(key)
    params = es.default_params
    args = inspect.signature(es.init).parameters

    if "population" in args:  # starts from a population of candidates
        Z0 = space.sample_normalized(np.random.default_rng(seed), size=es.population_size)
        state = es.init(key_init, jnp.asarray(Z0, dtype=jnp.float32), jnp.asarray(losses_for(Z0)), params)
    elif "means" in args:  # starts from several means
        means = jnp.zeros((es.num_populations, space.dim), dtype=jnp.float32)
        state = es.init(key_init, means, params)
    else:  # starts from one mean (distribution-based: CMA_ES, xNES, EvoTF_ES, ...)
        state = es.init(key_init, jnp.zeros((space.dim,), dtype=jnp.float32), params)
    return state, params, key


es = make_strategy(STRATEGY, space.dim, POP_SIZE)
print("strategy        :", type(es).__name__)
print("population_size :", es.population_size, "| dims:", es.num_dims)

## 6. Run the search

That is the whole loop — the same four steps whether `STRATEGY` is CMA-ES,
PSO or anything else:

```text
ask  ->  get POP_SIZE candidate vectors from the algorithm
clip ->  keep them inside [-1, 1]
evaluate -> score them with the simulation (one simulation each)
tell ->  report the scores back; the algorithm updates its search
```

We also keep our own best-so-far, because the notebook should work the same way
for every strategy.

In [ ]:
state, params, key = start(es, SEED)

history = []  # best fitness per generation
best_z, best_fitness = None, -np.inf

for generation in range(1, GENERATIONS + 1):
    key, key_ask, key_tell = jax.random.split(key, 3)

    Z, state = es.ask(key_ask, state, params)
    Z = jnp.asarray(np.clip(np.asarray(Z, dtype=float), -1.0, 1.0), dtype=jnp.float32)

    fitness = -losses_for(Z)  # evosax asked for losses; we keep fitness
    state, _info = es.tell(key_tell, Z, jnp.asarray(-fitness, dtype=jnp.float32), state, params)

    i = int(np.argmax(fitness))
    if fitness[i] > best_fitness:
        best_z, best_fitness = np.asarray(Z[i], dtype=float), float(fitness[i])
    history.append(best_fitness)
    print(f"generation {generation}: best fitness {best_fitness:+.4f}  (mean {fitness.mean():+.4f})")

best_genome = to_genome(best_z)
final = evaluator.evaluate(best_genome)  # one more simulation for the two parts

print()
print("best fold angles (radians):", np.round(best_genome, 4).tolist())
print(f"best fitness              : {final.fitness:+.4f}")
print(f"  movement part (m)       : {final.descriptors['displacement_x']:+.4f}")
print(f"  fold part (1.0 = perfect): {final.descriptors['fold_score']:.4f}")
print(f"  mean hinge error (deg)  : {final.descriptors['fold_mean_abs_error_deg']:.2f}")
print(f"best fitness per generation: {[round(x, 4) for x in history]}")

### What do these numbers mean?

* **movement part** — how far the sheet's centre slid, in metres. It can be
  **negative** if the paper moved backwards; that is a real outcome, not a bug.
* **fold part** — `0` means the paper is still flat, `1` means every hinge is
  within a couple of degrees of its target angle.
* **fitness** — the weighted mix you chose in cell 0. A small or negative value
  is normal for a short run; the *trend across generations* is what matters.

A short run of a few generations on a CPU mostly shows that the wiring works.
Real searches need a GPU runtime, a larger budget and the packaged 4×4 pattern.

## 8. Save the design

`save_genome` writes a plain crease-pattern JSON you can inspect, share or open
in any origami tool; `save_best` stores the same design plus its fitness, so a
later run can warm-start from it.

In [ ]:
crease_path = save_genome(best_genome, space, "runs/evosax_best_pattern.json")
best_path = save_best(
    best_genome,
    space,
    "runs/evosax_best.json",
    fitness=final.fitness,
    extra={"strategy": STRATEGY, "movement_weight": MOVEMENT_WEIGHT, "fold_weight": FOLD_WEIGHT},
)
print("crease pattern :", crease_path)
print("checkpoint     :", best_path)

## 9. Try other algorithms

Change `STRATEGY` in cell 0 and run the notebook again from that cell. Here is a
ready-made comparison you can also paste into one cell — it runs a few
strategies against the *same* objective so the scores are directly comparable:

```python
for name in ["CMA_ES", "Open_ES", "xNES", "PSO", "SimpleGA"]:
    es = make_strategy(name, space.dim, POP_SIZE)
    state, params, key = start(es, SEED)
    best = -np.inf
    for _ in range(GENERATIONS):
        key, k_ask, k_tell = jax.random.split(key, 3)
        Z, state = es.ask(k_ask, state, params)
        Z = jnp.asarray(np.clip(np.asarray(Z, dtype=float), -1, 1), dtype=jnp.float32)
        f = -losses_for(Z)
        state, _ = es.tell(k_tell, Z, jnp.asarray(-f, dtype=jnp.float32), state, params)
        best = max(best, float(f.max()))
    print(f"{name:24s} best={best:+.4f}")
```

Rules of thumb:

| Situation | Good first choice |
|---|---|
| Few dimensions (< 20), need reliability | `CMA_ES` |
| Very large populations / GPU | `Open_ES` |
| Discrete or combinatorial tricks | `SimpleGA` |
| You want an uncomplicated baseline | `ARS`, `PGPE`, `RandomSearch` |

> ### Honest cost note
>
> Every fitness value is one **full physics simulation**. On a CPU a short run
> is seconds per simulation, so `POP_SIZE × GENERATIONS` is the real cost and
> `evosax` itself is almost free. For a real search use a GPU runtime, raise the
> budget, and try the packaged 4×4 pattern
> (`DesignSpace.from_default_pattern()`, 15 variables).

> ### Folding weights
>
> `70 / 30` is a friendly default: movement keeps the search exploring, folding
> keeps the design honest. If your goal *is* a well-folded shape, move towards
> `30 / 70` or `0 / 100`. See `docs/FITNESS.md` for the exact formulas.

Next: **01** (black-box optimizers), **02** (every `evosax` strategy),
**03** (reinforcement learning), **04** (an AI agent in the loop).

Docs: `docs/OPTIMIZATION_ES.md` · `docs/FITNESS.md` · `docs/INTEGRATIONS.md`.
Main repository: [github.com/ihsan31415/origami-engine-mjx](https://github.com/ihsan31415/origami-engine-mjx).